In [1]:
import boto3
from brollm import BaseContract
from typing import Any

REGION = "us-east-1"
MODEL_ID = "qwen.qwen3-vl-235b-a22b"

client = boto3.client("bedrock-runtime", region_name=REGION)

def input_fn(
    system_prompt: str,
    messages: list[dict],
    tool_registry: list[dict] | None = None,
) -> dict:
    kwargs = {
        "modelId": MODEL_ID,
        "system": [{"text": system_prompt}],
        "messages": messages,
    }
    if tool_registry:
        kwargs["toolConfig"] = {"tools": tool_registry}
    return client.converse(**kwargs)

def output_fn(response: dict) -> Any:
    return response

llm = BaseContract(input_fn=input_fn, output_fn=output_fn)

In [2]:
tool_registry = [
    {
        "toolSpec": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                }
            },
        }
    },
]

reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": [{"text": "What's the weather in Paris?"}]}],
    tool_registry=tool_registry,
)

reply

{'ResponseMetadata': {'RequestId': '67552801-5c7c-4e81-b441-9fcc3acae638',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 12 Sep 2026 05:05:51 GMT',
   'content-type': 'application/json',
   'content-length': '297',
   'connection': 'keep-alive',
   'x-amzn-requestid': '67552801-5c7c-4e81-b441-9fcc3acae638'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'toolUse': {'toolUseId': 'tooluse_6toADApfEuhgoRjuFPhRZf',
      'name': 'get_weather',
      'input': {'city': 'Paris'}}}]}},
 'stopReason': 'tool_use',
 'usage': {'inputTokens': 163, 'outputTokens': 20, 'totalTokens': 183},
 'metrics': {'latencyMs': 531}}

In [3]:
reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=[{"role": "user", "content": [{"text": "What's 12 * 7?"}]}],
    tool_registry=tool_registry,
)

reply

{'ResponseMetadata': {'RequestId': '76587ffe-bcdf-4313-b47a-b6041d1c8bff',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 12 Sep 2026 05:05:56 GMT',
   'content-type': 'application/json',
   'content-length': '216',
   'connection': 'keep-alive',
   'x-amzn-requestid': '76587ffe-bcdf-4313-b47a-b6041d1c8bff'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': '12 * 7 = 84'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 165, 'outputTokens': 10, 'totalTokens': 175},
 'metrics': {'latencyMs': 4477}}

In [6]:
tool_registry = [
    {
        "toolSpec": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {"city": {"type": "string"}},
                    "required": ["city"],
                }
            },
        }
    },
    {
        "toolSpec": {
            "name": "convert_currency",
            "description": "Convert an amount from one currency to another",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "amount": {"type": "number"},
                        "from_currency": {"type": "string"},
                        "to_currency": {"type": "string"},
                    },
                    "required": ["amount", "from_currency", "to_currency"],
                }
            },
        }
    },
]

messages = [
    {
        "role": "user",
        "content": [{"text": "What's the weather in Tokyo, and how much is 100 USD in JPY?"}],
    }
]

reply = llm(
    system_prompt="You are a helpful assistant.",
    messages=messages,
    tool_registry=tool_registry,
)

reply

{'ResponseMetadata': {'RequestId': '027ef501-278d-4c46-a86d-7d5b3ff44cba',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 12 Sep 2026 05:07:08 GMT',
   'content-type': 'application/json',
   'content-length': '448',
   'connection': 'keep-alive',
   'x-amzn-requestid': '027ef501-278d-4c46-a86d-7d5b3ff44cba'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'toolUse': {'toolUseId': 'tooluse_kb2ggV37YSJlObk4TA7Jru',
      'name': 'get_weather',
      'input': {'city': 'Tokyo'}}},
    {'toolUse': {'toolUseId': 'tooluse_n43wy3Whm5LkL3v9padmoX',
      'name': 'convert_currency',
      'input': {'amount': 100,
       'from_currency': 'USD',
       'to_currency': 'JPY'}}}]}},
 'stopReason': 'tool_use',
 'usage': {'inputTokens': 267, 'outputTokens': 58, 'totalTokens': 325},
 'metrics': {'latencyMs': 1203}}

In [ ]:
results = {
    "get_weather": "22°C and sunny in Tokyo",
    "convert_currency": "100 USD = 15,050 JPY",
}


information = "\n".join([v for k,v in results.items()])
messages[-1]['content'][0]['text'] = f"{messages[-1]}\n\nFYI:\n{information}"

final = llm(
    system_prompt="You are a helpful assistant.",
    messages=messages,
)
# 79, 35 without tool
final

{'ResponseMetadata': {'RequestId': 'b0d043ab-e948-4174-9407-e64db18b2281',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 12 Sep 2026 05:07:10 GMT',
   'content-type': 'application/json',
   'content-length': '308',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'b0d043ab-e948-4174-9407-e64db18b2281'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': 'The weather in Tokyo is currently 22°C and sunny.  \nAdditionally, 100 USD is equivalent to 15,050 JPY.'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 79, 'outputTokens': 35, 'totalTokens': 114},
 'metrics': {'latencyMs': 1813}}

In [ ]:
# assistant_message = reply["output"]["message"]  # role="assistant", content=[{"toolUse": {...}}, ...]
# messages.append(assistant_message)

# tool_uses = [block["toolUse"] for block in assistant_message["content"] if "toolUse" in block]

# results = {
#     "get_weather": "22°C and sunny in Tokyo",
#     "convert_currency": "100 USD = 15,050 JPY",
# }

# # Bedrock requires every toolResult bundled into ONE user message, each tied
# # back to its toolUseId (no equivalent of Ollama's by-name tool_name matching).
# tool_result_content = [
#     {
#         "toolResult": {
#             "toolUseId": tu["toolUseId"],
#             "content": [{"text": results[tu["name"]]}],
#         }
#     }
#     for tu in tool_uses
# ]

# messages.append({"role": "user", "content": tool_result_content})

# final = llm(
#     system_prompt="You are a helpful assistant.",
#     messages=messages,
#     tool_registry=tool_registry,
# )
# # 363, 34 with tool
# final

{'ResponseMetadata': {'RequestId': '7dadc700-ac80-417f-b649-352d0a46d116',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 12 Sep 2026 04:57:09 GMT',
   'content-type': 'application/json',
   'content-length': '303',
   'connection': 'keep-alive',
   'x-amzn-requestid': '7dadc700-ac80-417f-b649-352d0a46d116'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': 'The current weather in Tokyo is 22°C and sunny. Additionally, 100 USD is equivalent to 15,050 JPY.'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 363, 'outputTokens': 34, 'totalTokens': 397},
 'metrics': {'latencyMs': 765}}